In [1]:
def clean_and_tokenize(text):
    """Removes punctuation, converts to lowercase, and tokenizes text."""
    if isinstance(text, str):
        text = text.translate(str.maketrans('', '', string.punctuation))
        text = text.lower()
        return text.split()
    return [] # Return empty list for non-string inputs

In [2]:
import os; import sys; import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

out_dir = '/content/drive/MyDrive/NLPforFakeNews'
os.makedirs(out_dir, exist_ok=True)

# Add this directory to Python's search path
sys.path.append(out_dir)

#Save the datasets
proc_dir = os.path.join(out_dir, "processed_datasets")
os.makedirs(proc_dir, exist_ok=True)

# Define file paths inside that folder
train_path = os.path.join(proc_dir, "train_FvT.csv")
val_path   = os.path.join(proc_dir, "val_FvT.csv")
test_path  = os.path.join(proc_dir, "test_FvT.csv")

# Read the datasets
train_df = pd.read_csv(train_path)
val_df   = pd.read_csv(val_path)
test_df  = pd.read_csv(test_path)
train_df.dropna(subset=['text_full'], inplace=True)
val_df.dropna(subset=['text_full'], inplace=True)
test_df.dropna(subset=['text_full'], inplace=True)

Mounted at /content/drive


In [3]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
import pandas as pd
import string

stop_words = ["i", "me", "my", "myself", "we", "our", "ours", "ourselves", "you", "your", "yours", "yourself",
              "yourselves", "he", "him", "his", "himself", "she", "her", "hers", "herself", "it", "its", "itself",
              "they", "them", "their", "theirs", "themselves", "what", "which", "who", "whom", "this", "that",
              "these", "those", "am", "is", "are", "was", "were", "be", "been", "being", "have", "has", "had",
              "having", "do", "does", "did", "doing", "a", "an", "the", "and", "but", "if", "or", "because",
              "as", "until", "while", "of", "at", "by", "for", "with", "about", "against", "between", "into",
              "through", "during", "before", "after", "above", "below", "to", "from", "up", "down", "in", "out",
              "on", "off", "over", "under", "again", "further", "then", "once", "here", "there", "when", "where",
              "why", "how", "all", "any", "both", "each", "few", "more", "most", "other", "some", "such", "no",
              "nor", "not", "only", "own", "same", "so", "than", "too", "very", "s", "t", "can", "will", "just",
              "don", "should", "now"]

# Set a random seed for reproducibility
np.random.seed(42)

# Combine train and validation data for GridSearchCV
train_val_df = pd.concat([train_df, val_df], ignore_index=True)

X_train_val = train_val_df['text_full'].values
y_train_val = train_val_df['label'].values
X_test = test_df['text_full'].values
y_test = test_df['label'].values

# Create a custom cross-validation split using the combined data
# The first split uses the original training data as the training fold
# and the original validation data as the validation fold.
train_indices = train_df.index.tolist()
val_indices = [i + len(train_df) for i in val_df.index.tolist()] # Adjust indices for the combined df

custom_cv = [(train_indices, val_indices)]


tfidf = TfidfVectorizer(strip_accents=None,
                        lowercase=False,
                        preprocessor=None)

small_param_grid = [
{
  'vect__ngram_range': [(1, 1),(1,2),(1,3)],
  'vect__stop_words': [stop_words, None],
  'vect__tokenizer': [clean_and_tokenize],
  'clf__penalty': ['l1','l2'],
  'clf__C': [1.0, 5.0, 10.0]
},
]
lr_tfidf = Pipeline([
  ('vect', tfidf),
  ('clf', LogisticRegression(solver='liblinear'))
])

gs_lr_tfidf = GridSearchCV(lr_tfidf, small_param_grid,
scoring='accuracy', cv=custom_cv, # Use the custom cross-validation split
verbose=2, n_jobs=-1)

gs_lr_tfidf.fit(X_train_val, y_train_val) # Fit on the combined train/validation data

print(f'Best parameter set: {gs_lr_tfidf.best_params_}')
print(f'Validation Accuracy: {gs_lr_tfidf.best_score_:.3f}')

# Evaluate on the test set
clf = gs_lr_tfidf.best_estimator_
print(f'Test Accuracy: {clf.score(X_test, y_test):.3f}')

Fitting 1 folds for each of 36 candidates, totalling 36 fits


KeyboardInterrupt: 

In [ ]:
# Combine train and validation data for GridSearchCV
train_val_df = pd.concat([train_df, val_df], ignore_index=True)

X_train_val = train_val_df['text_full'].values
y_train_val = train_val_df['label'].values
X_test = test_df['text_full'].values
y_test = test_df['label'].values

# Create a custom cross-validation split using the combined data
# The first split uses the original training data as the training fold
# and the original validation data as the validation fold.
train_indices = train_df.index.tolist()
val_indices = [i + len(train_df) for i in val_df.index.tolist()] # Adjust indices for the combined df

custom_cv = [(train_indices, val_indices)]